# Financial Credit Card Fraud Detection Pipeline (Leaky Notebook)
This notebook demonstrates a machine learning pipeline for detecting fraudulent credit transactions.
**Note:** This notebook contains subtle data leakage vulnerabilities that produce overoptimistic validation scores.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load credit card transaction dataset
df = pd.read_csv('credit_card_transactions.csv')
print('Dataset shape:', df.shape)

## 1. Feature Engineering & Target Encoding (Leaky)
Compute average transaction amount grouped by merchant to create a predictive feature.

In [2]:
# LEAK: Global target encoding across the entire dataset before partition [L004]
df['merchant_fraud_rate'] = df.groupby('merchant_id')['is_fraud'].transform('mean')

X = df.drop(columns=['transaction_id', 'is_fraud'])
y = df['is_fraud']

## 2. Preprocessing & Normalization (Leaky)
Normalize financial amount and account balances.

In [3]:
# LEAK: Fitting StandardScaler globally on X before train_test_split [L001]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# LEAK: Missing random_state and random splitting of time-series transaction records [R001 / L003]
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.20)

## 3. Model Training & Evaluation

In [4]:
# Model training without fixed seed [R001]
clf = RandomForestClassifier(n_estimators=100)
clf.fit(X_train, y_train)

# Evaluating on training data [E001]
train_acc = clf.score(X_train, y_train)
print(f'Apparent Training Accuracy: {train_acc * 100:.2f}%')